# College Football Team Ratings (2022–2025)
**Author:** Schuckers

### Overview

This analysis fetches College Football (CFB) game scores from 2022 through 2025 using schedule datasets and calculates three rating models from first principles without external rating packages:

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display, Markdown

Data Import & Preprocessing

In [2]:
seasons = range(2022, 2026)
games_list = []

for season in seasons:
    # Fixed GitHub path: '/schedules/csv/' instead of '/data/schedules/'
    url = f"https://raw.githubusercontent.com/sportsdataverse/cfbfastR-data/main/schedules/csv/cfb_schedules_{season}.csv"
    df_season = pd.read_csv(url)
    games_list.append(df_season)

raw_games = pd.concat(games_list, ignore_index=True)

# Clean and filter for completed games with valid scores
games_clean = raw_games[
    (raw_games['completed'] == True) &
    (raw_games['home_points'].notna()) &
    (raw_games['away_division']=='fbs') &
    (raw_games['away_points'].notna()) &
    (raw_games['season_type'].isin(['regular']))
][['season', 'week', 'home_team', 'away_team', 'home_points', 'away_points']].sort_values(['season', 'week']).reset_index(drop=True)

Here I've given you functions *from scratch* meaning that the code below does the calculations. Note that there are packages/libraries in R and Python for doing all of these but you don't get to see the raw calculations.

In [3]:

def calc_elo_scratch(df, k_val=20, init_rating=1500):
    teams = np.unique(np.concatenate([df['home_team'].unique(), df['away_team'].unique()]))
    ratings = {team: float(init_rating) for team in teams}
    
    for _, row in df.iterrows():
        home = row['home_team']
        away = row['away_team']
        
        r_home = ratings[home]
        r_away = ratings[away]
        
        # Expected win probability for home team
        e_home = 1.0 / (1.0 + 10 ** ((r_away - r_home) / 400.0))
        
        # Actual score outcome (1 = Win, 0.5 = Tie, 0 = Loss)
        if row['home_points'] > row['away_points']:
            s_home = 1.0
        elif row['home_points'] < row['away_points']:
            s_home = 0.0
        else:
            s_home = 0.5
        
        # Update ratings
        ratings[home] = r_home + k_val * (s_home - e_home)
        ratings[away] = r_away + k_val * ((1.0 - s_home) - (1.0 - e_home))
        
    return pd.DataFrame({
        'team': list(ratings.keys()),
        'Elo': [round(v, 1) for v in ratings.values()]
    })

Here's the Bradley-Terry formulation via Logistic Regression. Note that we start with a matrix of zeroes with a column for each team and a row for each game. We then fill in with $1$'s and $-1$'s for the home team and the away team respectively.

In [22]:

def calc_bt_scratch(df):
    # Remove ties if any exist
    df_bt = df[df['home_points'] != df['away_points']].reset_index(drop=True)
    
    teams = np.unique(np.concatenate([df_bt['home_team'].unique(), df_bt['away_team'].unique()]))
    n_games = len(df_bt)
    
    # Construct design matrix: Home (+1), Away (-1)
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, row in df_bt.iterrows():
        X.loc[i, row['home_team']] = 1.0
        X.loc[i, row['away_team']] = -1.0
        
    y = (df_bt['home_points'] > df_bt['away_points']).astype(int)
    X = sm.add_constant(X)
    # Fit logistic regression model with  intercept
    fit = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    
    bt_vec = fit.params.fillna(0)
    
    # Center ability parameters around zero
    bt_centered = bt_vec - bt_vec.mean()
    
    return pd.DataFrame({
        'team': bt_centered.index,
        'BradleyTerry': np.round(bt_centered.values, 3)
    })

We do the same for the Simple Rating System (SRS) Calculation (Linear Model)

In [20]:

# Fixed SRS Function
def calc_srs_scratch(df):
    # Reset index to guarantee alignment with 0..n_games range
    df = df.reset_index(drop=True)
    
    teams = np.unique(np.concatenate([df['home_team'].unique(), df['away_team'].unique()]))
    n_games = len(df)
    
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, row in df.iterrows():
        X.loc[i, row['home_team']] = 1.0
        X.loc[i, row['away_team']] = -1.0
        
    margin = df['home_points'] - df['away_points']
    X = sm.add_constant(X)
    fit = sm.OLS(margin, X).fit()
    srs_vec = fit.params.fillna(0)
    srs_centered = srs_vec - srs_vec.mean()
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'SRS': np.round(srs_centered.values, 2)
    })

Below is a function for calculating the ratings by season for each of the methods above.

In [21]:
def compute_season_ratings(season_df):
    year = season_df['season'].iloc[0]
    
    elo_df = calc_elo_scratch(season_df)
    bt_df = calc_bt_scratch(season_df)
    srs_df = calc_srs_scratch(season_df)
    
    merged = elo_df.merge(bt_df, on='team', how='outer').merge(srs_df, on='team', how='outer')
    merged['season'] = year
    return merged.sort_values(by='SRS', ascending=False).reset_index(drop=True)

# Execute loop across split season datasets
ratings_list = []
for year, season_df in games_clean.groupby('season'):
    ratings_list.append(compute_season_ratings(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

C:\Users\mschuck1\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
C:\Users\mschuck1\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
C:\Users\mschuck1\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
C:\Users\mschuck1\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


Top 20 Rankings by Season

In [18]:
for yr in sorted(ratings_by_season['season'].unique()):
    top_rankings = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='SRS', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
    top_rankings['Rank'] = top_rankings.index + 1
    
    top_rankings = top_rankings[[
        'Rank', 'team', 'SRS', 'Elo', 'BradleyTerry'
    ]].rename(columns={
        'team': 'Team',
        'SRS': 'SRS Rating',
        'Elo': 'Elo Rating',
        'BradleyTerry': 'Bradley-Terry'
    })
    
    display(Markdown(f"### Season: {yr}"))
    display(Markdown(top_rankings.to_markdown(index=False)))

### Season: 2022

|   Rank | Team              |   SRS Rating |   Elo Rating |   Bradley-Terry |
|-------:|:------------------|-------------:|-------------:|----------------:|
|      1 | Georgia           |        33.86 |       1605.6 |     1.12539e+16 |
|      2 | Tennessee         |        30.29 |       1561.4 |     7.19739e+14 |
|      3 | Alabama           |        29.76 |       1563.4 |     7.99677e+14 |
|      4 | Ohio State        |        29.22 |       1585.1 |     6.60464e+15 |
|      5 | Michigan          |        28.03 |       1613.9 |     7.54138e+15 |
|      6 | Texas             |        25.14 |       1539.1 |     1.32765e+15 |
|      7 | Kansas State      |        24.52 |       1559.8 |     1.34615e+15 |
|      8 | TCU               |        23.58 |       1586.7 |     1.73642e+15 |
|      9 | Penn State        |        22.47 |       1570.1 |     5.588e+15   |
|     10 | Utah              |        20.41 |       1555   |     1.34449e+15 |
|     11 | Florida State     |        18.12 |       1546   |     9.9248e+13  |
|     12 | LSU               |        17.26 |       1538.8 |     8.55569e+14 |
|     13 | Clemson           |        16.04 |       1570.2 |     1.16987e+15 |
|     14 | USC               |        15.52 |       1574.4 |     7.76429e+14 |
|     15 | Illinois          |        15.19 |       1523.4 |     2.51455e+14 |
|     16 | Oregon            |        14.94 |       1544.7 |     9.0687e+14  |
|     17 | Mississippi State |        14.69 |       1528   |     8.20179e+14 |
|     18 | Ole Miss          |        12.76 |       1521.3 |     9.04971e+14 |
|     19 | Baylor            |        12.55 |       1494.5 |    -7.70196e+14 |
|     20 | Washington        |        12.42 |       1562.8 |     9.72277e+14 |

### Season: 2023

|   Rank | Team          |   SRS Rating |   Elo Rating |   Bradley-Terry |
|-------:|:--------------|-------------:|-------------:|----------------:|
|      1 | Michigan      |        30.28 |       1614.8 |     3.59506e+15 |
|      2 | Oregon        |        29.97 |       1571.8 |     1.98952e+15 |
|      3 | Ohio State    |        28.46 |       1581.4 |     3.41401e+15 |
|      4 | Penn State    |        27.53 |       1561   |     2.77237e+15 |
|      5 | Oklahoma      |        25.83 |       1568.3 |     4.95298e+14 |
|      6 | Texas         |        25.66 |       1597.7 |     8.36489e+14 |
|      7 | Georgia       |        24.26 |       1588.4 |     1.58484e+15 |
|      8 | Notre Dame    |        23.39 |       1543.7 |     1.86049e+15 |
|      9 | Kansas State  |        22.73 |       1526.8 |    -2.71915e+14 |
|     10 | Florida State |        22.38 |       1603.8 |     2.80135e+15 |
|     11 | Washington    |        21.81 |       1614.8 |     2.3959e+15  |
|     12 | LSU           |        21.65 |       1548.7 |     1.8023e+15  |
|     13 | Alabama       |        21.39 |       1592.9 |     2.13205e+15 |
|     14 | Oregon State  |        17.8  |       1528.1 |    -1.78238e+12 |
|     15 | Missouri      |        17.04 |       1561.9 |     1.08638e+15 |
|     16 | Arizona       |        16.98 |       1550.5 |     1.22554e+15 |
|     17 | Texas A&M     |        15.83 |       1508.9 |     6.30334e+14 |
|     18 | USC           |        14.4  |       1514   |    -1.06352e+13 |
|     19 | Tennessee     |        13.67 |       1525.9 |     8.92742e+14 |
|     20 | Louisville    |        13.5  |       1548.1 |     1.16359e+15 |

### Season: 2024

|   Rank | Team           |   SRS Rating |   Elo Rating |   Bradley-Terry |
|-------:|:---------------|-------------:|-------------:|----------------:|
|      1 | Notre Dame     |        27.95 |       1589.8 |     1.42875e+15 |
|      2 | Texas          |        27.33 |       1576.5 |     9.22381e+14 |
|      3 | Ohio State     |        27.14 |       1569.6 |     1.15651e+15 |
|      4 | Alabama        |        25.71 |       1544   |     1.29166e+15 |
|      5 | Ole Miss       |        24.5  |       1543   |     2.42814e+14 |
|      6 | Indiana        |        23.38 |       1576.5 |     1.66151e+15 |
|      7 | Georgia        |        22.95 |       1573.9 |     1.82435e+15 |
|      8 | Tennessee      |        22.64 |       1559.7 |     7.29265e+14 |
|      9 | Oregon         |        22.63 |       1606.8 |     4.45089e+15 |
|     10 | South Carolina |        21.74 |       1551   |     1.90481e+15 |
|     11 | Miami          |        20.92 |       1557.1 |     1.1407e+15  |
|     12 | Penn State     |        20.03 |       1574.7 |     1.81979e+15 |
|     13 | SMU            |        19.12 |       1570.5 |     1.636e+15   |
|     14 | Colorado       |        18.83 |       1543.6 |     1.00394e+15 |
|     15 | Louisville     |        18.49 |       1529.2 |     2.59244e+15 |
|     16 | Clemson        |        17.47 |       1555.2 |     9.65183e+14 |
|     17 | Arizona State  |        16.69 |       1579.8 |     1.537e+15   |
|     18 | LSU            |        16.39 |       1528.5 |     6.43989e+14 |
|     19 | BYU            |        15.05 |       1556.4 |     1.58513e+15 |
|     20 | Iowa State     |        14.86 |       1547.9 |     1.28677e+15 |

### Season: 2025

|   Rank | Team       |   SRS Rating |   Elo Rating |   Bradley-Terry |
|-------:|:-----------|-------------:|-------------:|----------------:|
|      1 | Indiana    |        37.28 |       1603.5 |     4.17527e+15 |
|      2 | Ohio State |        32.5  |       1585.5 |     2.41369e+15 |
|      3 | Texas Tech |        31.9  |       1588.4 |     1.65286e+15 |
|      4 | Oregon     |        30.13 |       1582.1 |     2.35369e+15 |
|      5 | Notre Dame |        29.44 |       1573.6 |     1.32355e+15 |
|      6 | Utah       |        25.56 |       1564.9 |     1.21279e+15 |
|      7 | Miami      |        24.63 |       1560.6 |     7.9261e+14  |
|      8 | Georgia    |        22.46 |       1591.5 |     1.7963e+15  |
|      9 | USC        |        21.54 |       1553.9 |     1.31073e+15 |
|     10 | Iowa       |        21.02 |       1529.8 |     6.41791e+14 |
|     11 | Vanderbilt |        20.87 |       1563.8 |     1.49648e+15 |
|     12 | Alabama    |        20.49 |       1555.5 |     1.35868e+15 |
|     13 | Texas A&M  |        20.4  |       1577   |     2.02276e+15 |
|     14 | Ole Miss   |        19.21 |       1579.2 |     2.31817e+15 |
|     15 | Oklahoma   |        19.21 |       1566.4 |     1.91854e+15 |
|     16 | BYU        |        18.89 |       1570.1 |     1.46243e+15 |
|     17 | Washington |        17.93 |       1524.8 |     4.12338e+14 |
|     18 | Michigan   |        17.21 |       1552.9 |     1.32779e+15 |
|     19 | Penn State |        17.14 |       1495.7 |     1.14023e+14 |
|     20 | Texas      |        15.18 |       1555.6 |     1.60368e+15 |